# Train the V3 LSTM on Google Colab

Downloads 20 years of hourly ERA5 (2006-2025) for the project domain, reduces it to
domain means plus wind-driven temperature advection, trains the V3 model and exports
the weights the apps run on.

**Before you start**
- *Runtime → Change runtime type → T4 GPU* (CPU works too, just slower).
- A free [Copernicus CDS](https://cds.climate.copernicus.eu) account. Open the
  [ERA5 single-levels dataset](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download),
  accept the licence once at the bottom of the page, and copy your API key from your profile page.

**Colab sessions time out.** With `USE_DRIVE = True` the downloads, CSVs and model
checkpoints live in Google Drive, so after a disconnect you just run every cell again:
finished years are skipped, and `RESUME = True` continues training from the last checkpoint
with the same train/validation/test split.

| step | time |
|---|---|
| download + reduce, per year | 5-60 min (the CDS queue decides) |
| training, 60 epochs on a T4 | ~10-20 min |

In [ ]:
REPO = "https://github.com/Harshpreet-Singh07/First_TempModelLSTM.git"
BRANCH = "main"
USE_DRIVE = True        # keep data + checkpoints in Drive across sessions

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/era5_lstm"
else:
    WORK = "/content/era5_lstm"
os.makedirs(WORK, exist_ok=True)

%cd /content
!rm -rf repo && git clone -q -b {BRANCH} {REPO} repo
%cd /content/repo

# point the data and model folders at WORK, keeping anything already saved there
for sub in ["data/processed", "lstm/models", "lstm/output"]:
    keep = f"{WORK}/{sub}"
    os.makedirs(keep, exist_ok=True)
    !cp -rn {sub}/. {keep}/
    !rm -rf {sub} && ln -s {keep} {sub}
!ls data/processed lstm/models

In [ ]:
!pip -q install cdsapi cfgrib eccodes xarray

## CDS API key
Stored in `~/.cdsapirc` on this Colab machine only - it is not written to Drive or the repo.

In [ ]:
from getpass import getpass
from pathlib import Path

rc = Path.home() / ".cdsapirc"
if not rc.exists():
    key = getpass("CDS API key (from your CDS profile page): ").strip()
    rc.write_text(f"url: https://cds.climate.copernicus.eu/api
key: {key}
")
print("CDS key ready")

## 1. Download and reduce 20 years

One CDS request per year (2 m temperature, dewpoint, 10 m wind, precipitation over the
2025 domain). Each GRIB is reduced to `data/processed/domain_mean_hourly_<year>.csv` and
then deleted to save space. Re-running skips finished years.

In [ ]:
YEARS = "2006-2025"
!python scripts/era5_years.py --years {YEARS} --raw {WORK}/raw --delete-grib

In [ ]:
import pandas as pd
rows = []
for p in sorted(Path("data/processed").glob("domain_mean_hourly_*.csv")):
    d = pd.read_csv(p, index_col="valid_time", parse_dates=True)
    rows.append({"year": p.stem[-4:], "hours": len(d), "columns": len(d.columns),
                 "t2m mean": round(d.t2m.mean(), 2), "t2m_adv mean": round(d.get("t2m_adv", pd.Series(dtype=float)).mean(), 3)})
pd.DataFrame(rows)

## 2. Train

Defaults: 72 h lookback, 1 h ahead, inputs `t2m, t2m_d1, d2m, u10, v10, t2m_adv` plus time of
day/year, split by whole years 70/15/15 (train 2006-2019, validation 2020-2022, test 2023-2025).
Set `RESUME = True` to continue the saved model instead of starting over.

In [ ]:
NAME = "v3"
EPOCHS = 60
RESUME = False

flags = "--resume" if RESUME else ""
!python lstm/train_lstm.py --name {NAME} --years {YEARS} --epochs {EPOCHS} {flags}

In [ ]:
from IPython.display import Image, display
print(open(f"lstm/output/{NAME}/metrics.csv").read())
display(Image(f"lstm/output/{NAME}/01_loss_curve.png", width=500))
display(Image(f"lstm/output/{NAME}/02_test_predictions.png"))

## 3. Bring the model back to the repo

Downloads a zip with the model, its scaler, the numpy weights, the training figures and the
yearly CSVs. Unzip it into the project folder on your computer (the paths inside match the
repo), check the app with `python app.py`, then commit and push.

In [ ]:
!cd /content/repo && zip -qr /content/{NAME}_bundle.zip     lstm/models/lstm_{NAME}.keras lstm/models/lstm_{NAME}_weights.npz lstm/models/scaler_{NAME}.json     lstm/output/{NAME} data/processed
!ls -lh /content/{NAME}_bundle.zip
from google.colab import files
files.download(f"/content/{NAME}_bundle.zip")